# Syntetyczna historia awarii

Aplikacja "Głosowe zgłaszanie awarii" pokazuje podobne awarie z przeszłości. Prawdziwych zgłoszeń awarii nie można publikować, a publicznego zbioru awarii infrastruktury farmaceutycznej nie ma, więc historię przygotowałem sam:

1. generuję 150 awarii modelem `gpt-4o-mini` z `instructor`,
2. przeglądam, co wyszło,
3. redaguję rekordy,
4. ładuję gotową historię do Qdranta.

W folderze `data` są dwa pliki:
* `historia_awarii_surowa.json` - to, co oddał model, bez poprawek,
* `historia_awarii.json` - wersja po mojej redakcji. **To z niej korzysta aplikacja.**

## Instalujemy wymagane biblioteki (conda wymagana)

- otwórz terminal
- uruchom `conda activate od_zera_do_ai`
- uruchom `conda config --append channels conda-forge`
- uruchom `conda install -y qdrant-client openai python-dotenv pandas`
- uruchom `pip install instructor`

In [1]:
import json
import random
import shutil
from datetime import datetime, timedelta
from pathlib import Path

import instructor
import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PointStruct,
    VectorParams,
)

# Struktura zgłoszenia jest wspólna z aplikacją
from zgloszenie import AwariaHistoryczna, ticket_to_text

In [2]:
env = dotenv_values(".env")

HISTORY_MODEL = "gpt-4o-mini"

EMBEDDING_MODEL = "text-embedding-3-large"

EMBEDDING_DIM = 3072

QDRANT_COLLECTION_NAME = "awarie"

QDRANT_LOCAL_PATH = "qdrant_db"

RAW_HISTORY_PATH = Path("data") / "historia_awarii_surowa.json"

HISTORY_PATH = Path("data") / "historia_awarii.json"

# Ile awarii na jedno pytanie do modelu - przy większych porcjach zaczyna się powtarzać
BATCH_SIZE = 10

# Rekordy historii dostają numery od miliona w górę. Dzięki temu nie mieszają się
# z numerami zgłoszeń z aplikacji, a ponowny import nadpisuje te same punkty.
HISTORY_ID_OFFSET = 1_000_000


def get_openai_client():
    return OpenAI(api_key=env["OPENAI_API_KEY"])

## 1. Generowanie

Dla każdego systemu ustalam, ile awarii chcę, i podpowiadam modelowi typowe tematy awarii w utrzymaniu ruchu, żeby historia nie była oderwana od życia.

In [3]:
PLAN = {
    "HVAC": (
        40,
        "centrale wentylacyjne, wentylatory i falowniki, filtry wstępne i HEPA, nagrzewnice i chłodnice, "
        "nawilżacze parowe, przepustnice, różnice ciśnień i kaskady ciśnień w pomieszczeniach czystych, "
        "temperatura i wilgotność, agregaty wody lodowej, wyciągi procesowe",
    ),
    "Woda oczyszczona": (
        30,
        "stacja uzdatniania (zmiękczacze, odwrócona osmoza, EDI), zbiornik magazynowy, pętla dystrybucyjna, "
        "pompy obiegowe, lampy UV, ozonowanie, sanityzacja termiczna, przewodność, TOC, temperatura i przepływ "
        "w pętli, zawory membranowe w punktach poboru, wymienniki ciepła",
    ),
    "Izolatory": (
        25,
        "izolatory do pracy z substancjami wysoko aktywnymi (HPAPI), rękawice i mankiety, test szczelności, "
        "podciśnienie w komorze, filtry HEPA push-push, śluzy materiałowe i porty RTP, system mycia WIP, "
        "blokady drzwi, czujniki ciśnienia, worki bezpiecznej wymiany",
    ),
    "Automatyka": (
        25,
        "sterowniki PLC, panele HMI, system BMS, system monitoringu środowiska (EMS), czujniki temperatury, "
        "wilgotności i ciśnienia, komunikacja sieciowa, siłowniki zaworów i przepustnic, kalibracja czujników, "
        "archiwizacja danych i alarmów, zasilacze 24 V",
    ),
    "Sprężone powietrze i gazy": (
        15,
        "sprężarki bezolejowe, osuszacze i punkt rosy, filtry końcowe, zbiorniki buforowe, spadki ciśnienia "
        "w sieci, nieszczelności, instalacja azotu, reduktory i zawory bezpieczeństwa",
    ),
    "Zasilanie": (
        15,
        "rozdzielnice, UPS-y i baterie, agregat prądotwórczy, automatyka SZR, zaniki i wahania napięcia, "
        "wyłączniki i zabezpieczenia, falowniki, kompensacja mocy biernej",
    ),
}

sum(ile for ile, _ in PLAN.values())

150

Generuję przez `instructor`, tak jak w lekcji text-to-json (moduł 8, lekcja 15). Trzy rzeczy zrobiłem inaczej niż w kursie:

* używam tańszego `gpt-4o-mini`, bo rekordów jest dużo,
* daję `temperature=1`, bo chcę różnorodnych awarii, a nie w kółko tej samej,
* włączam tryb `TOOLS_STRICT` - bez niego `gpt-4o-mini` gubił pojedyncze pola w liście awarii i walidacja Pydantic się wywalała.

In [4]:
# instructor zwraca jeden obiekt, więc listę awarii opakowuję w klasę
# (tak jak chapters_overview w klasie ShortStory z lekcji 15)
class ListaAwarii(BaseModel):
    awarie: list[AwariaHistoryczna]


def generate_batch(system, tematy, ile, juz_sa):
    instructor_openai_client = instructor.from_openai(
        get_openai_client(),
        mode=instructor.Mode.TOOLS_STRICT,
    )
    juz_sa_tekst = "\n".join(f"- {opis}" for opis in juz_sa) or "- (jeszcze nic)"
    res = instructor_openai_client.chat.completions.create(
        model=HISTORY_MODEL,
        temperature=1,
        response_model=ListaAwarii,
        messages=[
            {
                "role": "user",
                "content": f"""
Wymyśl {ile} realistycznych, zmyślonych awarii infrastruktury technicznej w zakładzie farmaceutycznym.
Wszystkie dotyczą systemu: {system}.
Tematy do wykorzystania: {tematy}.

Zasady:
- każda awaria ma być inna: inne urządzenie, inny objaw, inna przyczyna,
- objaw pisz tak, jak opisałby go technik na miejscu, a nie jak w podręczniku,
- kod alarmu podaj w mniej więcej połowie awarii (np. A112, F0031, AL-27), w pozostałych zostaw puste,
- lokalizacja zmyślona i ogólna, np. "budynek B, poziom 1, pomieszczenie techniczne 1.14",
- urządzenia nazywaj ogólnie z numerem, np. "centrala wentylacyjna AHU-04", bez nazw firm i producentów,
- priorytet i wpływy dobierz rozsądnie: większość awarii jest zwykła, krytycznych jest mało,
- przyczyna i działanie naprawcze konkretne, takie, żeby następny technik miał z nich pożytek,
- czas przestoju w godzinach, od 0 (bez przestoju) do kilkudziesięciu,
- braki_informacji zostaw puste,
- pisz po polsku.

Te awarie już mam, nie powtarzaj ich:
{juz_sa_tekst}
                """,
            },
        ],
    )

    return [awaria.model_dump() for awaria in res.awarie]


def generate_history():
    # Stałe ziarno, żeby daty wychodziły powtarzalnie
    random.seed(8)
    historia = []
    for system, (ile_razem, tematy) in PLAN.items():
        awarie_systemu = []
        while len(awarie_systemu) < ile_razem:
            ile = min(BATCH_SIZE, ile_razem - len(awarie_systemu))
            juz_sa = [f'{a["urzadzenie"]}: {a["objaw"]}' for a in awarie_systemu]
            nowe = generate_batch(system, tematy, ile, juz_sa)
            awarie_systemu.extend(nowe[:ile])
            print(f"{system}: {len(awarie_systemu)}/{ile_razem}")

        for awaria in awarie_systemu:
            # System i źródło ustawiam sam, żeby model nic tu nie pomieszał.
            # Datę losuję z ostatnich dwóch lat.
            awaria["system"] = system
            awaria["braki_informacji"] = []
            awaria["zrodlo"] = "historia"
            dni_temu = random.randint(1, 730)
            awaria["data"] = (datetime.now() - timedelta(days=dni_temu)).isoformat(timespec="seconds")
            historia.append(awaria)

    return historia

Generowanie kosztuje (około 20 zapytań do `gpt-4o-mini`) i za każdym razem daje inne awarie, dlatego robię je tylko raz. Jeśli surowy plik już jest, komórka niczego nie nadpisuje.

In [5]:
if RAW_HISTORY_PATH.exists():
    print(f"Surowa historia już jest w {RAW_HISTORY_PATH}, nie generuję jej drugi raz")
else:
    historia = generate_history()
    RAW_HISTORY_PATH.parent.mkdir(exist_ok=True)
    with open(RAW_HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(historia, f, ensure_ascii=False, indent=2)
    print(f"Zapisałem {len(historia)} awarii do {RAW_HISTORY_PATH}")

# Wersja do redakcji startuje jako kopia surowej i też nie jest nadpisywana
if not HISTORY_PATH.exists():
    shutil.copy(RAW_HISTORY_PATH, HISTORY_PATH)

Surowa historia już jest w data\historia_awarii_surowa.json, nie generuję jej drugi raz


## 2. Przegląd tego, co wygenerował model

In [6]:
with open(RAW_HISTORY_PATH, "r", encoding="utf-8") as f:
    surowa_df = pd.DataFrame(json.load(f))

surowa_df["system"].value_counts()

system
HVAC                         40
Woda oczyszczona             30
Izolatory                    25
Automatyka                   25
Sprężone powietrze i gazy    15
Zasilanie                    15
Name: count, dtype: int64

In [7]:
surowa_df[["urzadzenie", "objaw", "przyczyna", "dzialanie_naprawcze"]].sample(5, random_state=1)

,urzadzenie,objaw,przyczyna,dzialanie_naprawcze
14,filtr HEPA FH-06,"Podwyższone wartości cząstek w powietrzu, co m...",Filtr HEPA jest przestarzały i wymaga wymiany.,"Wymieniono filtr HEPA, przeprowadzono ponowną ..."
98,czujnik wilgotności-01,Czujnik wilgotności wyświetla nieprawidłowe wa...,Zanieczyszczenie czujnika wilgotności.,"Czujnik został oczyszczony, a jego kalibracja ..."
75,izolator HPAPI-06,"Niewłaściwy poziom podciśnienia w komorze, wed...","Uszkodzony czujnik ciśnienia, który nie przeka...",Wymieniono czujnik i przetestowano system pod ...
16,wentylator V-15,"Wentylator przestał działać, co prowadzi do wz...",Awaria silnika wentylatora.,"Wymieniono silnik wentylatora, przywrócono peł..."
131,filtr końcowy 3,"Na filtrze końcowym widoczne są ślady oleju, c...",Nieszczelność w połączeniach filtra z instalacją.,Dokonano regeneracji uszczelnienia i wymienion...


Po przeczytaniu wszystkich 150 rekordów wyszło, że nadają się na szkielet, ale nie na gotową historię:

* **błędy merytoryczne** - np. "niska przewodność niezgodna z normą" (niska przewodność wody to dobry wynik), nawilżacz parowy z "filtrem wstępnym", falownik "z paneli słonecznych",
* **przyczyna nie pasuje do naprawy** - np. przyczyna "zatkany filtr wstępny", a naprawa "wymieniono filtr HEPA",
* **niezgrabny język i literówki** - "diferenecja ciśnień", "wartości zaniku cząstek",
* **powtarzane kody alarmów** - A112 i F0031 w kilku różnych systemach (model przepisał przykłady z mojego polecenia),
* **zawyżone oceny** - wpływ na GMP "tak" w ponad połowie rekordów, także dla biur i szatni,
* **dziwne lokalizacje** - budynki od A do J, izolatory w pomieszczeniach biurowych.

In [8]:
# te same kody alarmów w wielu rekordach
kody = surowa_df[surowa_df["kod_alarmu"] != ""]["kod_alarmu"].value_counts()
kody[kody > 1]

kod_alarmu
F0031    8
A112     7
AL-27    6
AL-11    2
F001     2
Name: count, dtype: int64

## 3. Redakcja

Poprawiłem wszystkie 150 rekordów:

* objaw, działanie natychmiastowe, przyczynę i działanie naprawcze napisałem od nowa, językiem technika, tak żeby przyczyna i naprawa do siebie pasowały,
* priorytet oraz wpływ na bezpieczeństwo i GMP ustawiłem sam,
* każdy kod alarmu występuje tylko raz,
* poprawiłem dziwne nazwy urządzeń,
* budynki są tylko A-D, a izolatory stoją w pomieszczeniach produkcyjnych.

Bez zmian zostały: podział na systemy, daty i (poza pojedynczymi) czasy przestoju.

Poprawki trzymam w pliku `data/redakcja_historii.json`: dla każdego rekordu są tam tylko te pola, które zmieniłem. Komórka niżej nakłada je na surową historię i zapisuje wersję finalną, więc da się sprawdzić, co dokładnie zmieniłem, i odtworzyć wynik.

To dalej są dane zmyślone - do pokazania, jak działa wyszukiwanie podobnych awarii, a nie do wyciągania wniosków o prawdziwej instalacji.

In [9]:
REDACTION_PATH = Path("data") / "redakcja_historii.json"

with open(RAW_HISTORY_PATH, "r", encoding="utf-8") as f:
    surowa = json.load(f)

with open(REDACTION_PATH, "r", encoding="utf-8") as f:
    redakcja = json.load(f)

# surowy rekord + moje zmiany = rekord finalny
historia = []
for awaria, poprawka in zip(surowa, redakcja):
    historia.append({**awaria, **poprawka["zmiany"]})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(historia, f, ensure_ascii=False, indent=2)

print(f"Zapisałem {len(historia)} awarii po redakcji do {HISTORY_PATH}")

Zapisałem 150 awarii po redakcji do data\historia_awarii.json


In [10]:
# ile razy zmieniałem każde pole
pd.Series([pole for poprawka in redakcja for pole in poprawka["zmiany"]]).value_counts()

objaw                       150
dzialanie_natychmiastowe    150
przyczyna                   150
dzialanie_naprawcze         150
wplyw_na_gmp                 70
wplyw_na_bezpieczenstwo      67
lokalizacja                  44
priorytet                    43
urzadzenie                   38
kod_alarmu                   21
Name: count, dtype: int64

In [11]:
# przykładowy wpis z pliku redakcji
redakcja[55]

{'nr': 55,
 'urzadzenie_przed': 'stacja uzdatniania nr 3',
 'zmiany': {'objaw': 'Przewodność za stacją rośnie i zbliża się do limitu alarmowego.',
  'priorytet': 'wysoki',
  'wplyw_na_gmp': 'tak',
  'dzialanie_natychmiastowe': 'Sprawdziłem odczyty po kolei, za modułem EDI przewodność jest wysoka.',
  'przyczyna': 'Spadek sprawności modułu EDI.',
  'dzialanie_naprawcze': 'Zregenerowano moduł EDI, przewodność wróciła do normy.'}}

In [12]:
with open(HISTORY_PATH, "r", encoding="utf-8") as f:
    historia = json.load(f)

historia_df = pd.DataFrame(historia)
len(historia_df)

150

In [13]:
# przed i po redakcji
porownanie = {}
for kolumna in ["priorytet", "wplyw_na_bezpieczenstwo", "wplyw_na_gmp"]:
    porownanie[(kolumna, "surowa")] = surowa_df[kolumna].value_counts()
    porownanie[(kolumna, "po redakcji")] = historia_df[kolumna].value_counts()

pd.DataFrame(porownanie).fillna(0).astype(int)

priorytet             wplyw_na_bezpieczenstwo              \
             surowa po redakcji                  surowa po redakcji   
brak              0           0                      71         122   
krytyczny        17          11                       0           0   
możliwy           0           0                      51          14   
niski             7           9                       0           0   
tak               0           0                      28          14   
wysoki           51          50                       0           0   
średni           75          80                       0           0   

          wplyw_na_gmp              
                surowa po redakcji  
brak                10          26  
krytyczny            0           0  
możliwy             61          79  
niski                0           0  
tak                 79          45  
wysoki               0           0  
średni               0           0

In [14]:
# ten sam rekord przed i po redakcji
pola = ["urzadzenie", "objaw", "przyczyna", "dzialanie_naprawcze", "wplyw_na_gmp"]
pd.DataFrame({"surowa": surowa_df.loc[55, pola], "po redakcji": historia_df.loc[55, pola]})

,surowa,po redakcji
urzadzenie,stacja uzdatniania nr 3,stacja uzdatniania nr 3
objaw,"Monitor pokazuje niską wartość przewodności, j...",Przewodność za stacją rośnie i zbliża się do l...
przyczyna,"Problemy z filtracją wody, zanieczyszczenia pr...",Spadek sprawności modułu EDI.
dzialanie_naprawcze,"Sprawdzono i wymieniono filtry, przewodność wo...","Zregenerowano moduł EDI, przewodność wróciła d..."
wplyw_na_gmp,możliwy,tak


In [15]:
# po redakcji kody alarmów już się nie powtarzają
kody = historia_df[historia_df["kod_alarmu"] != ""]["kod_alarmu"].value_counts()
(kody > 1).sum()

0

## 4. Import do Qdranta

Połączenie i kolekcja tak samo jak w aplikacji (moduł 7, lekcje 11 i 24): gdy w `.env` są `QDRANT_URL` i `QDRANT_API_KEY`, łączę się z Qdrant Cloud, inaczej z bazą w folderze `qdrant_db`.

In [16]:
def get_qdrant_client():
    if env.get("QDRANT_URL") and env.get("QDRANT_API_KEY"):
        return QdrantClient(
            url=env["QDRANT_URL"],
            api_key=env["QDRANT_API_KEY"],
        )

    return QdrantClient(path=QDRANT_LOCAL_PATH)


qdrant_client = get_qdrant_client()

if not qdrant_client.collection_exists(QDRANT_COLLECTION_NAME):
    print("Tworzę kolekcję")
    qdrant_client.create_collection(
        collection_name=QDRANT_COLLECTION_NAME,
        vectors_config=VectorParams(
            size=EMBEDDING_DIM,
            distance=Distance.COSINE,
        ),
    )
else:
    print("Kolekcja już istnieje")

Kolekcja już istnieje


Embedding liczę z tego samego tekstu co w aplikacji (system, urządzenie, objaw, kod alarmu), żeby zgłoszenia i historia były porównywalne. Teksty wysyłam porcjami - `embeddings.create` przyjmuje listę (moduł 8, lekcja 16).

In [17]:
ticket_to_text(historia[0])

'System: HVAC. Urządzenie: centrala wentylacyjna AHU-01. Objaw: Centrala nie dochodzi do zadanego wydatku, na panelu przepływ nawiewu jest około 20% poniżej nastawy. Kod alarmu: AL-22'

In [18]:
def get_embeddings(texts):
    openai_client = get_openai_client()
    result = openai_client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL,
        dimensions=EMBEDDING_DIM,
    )

    return [d.embedding for d in result.data]


porcja = 50
for start in range(0, len(historia), porcja):
    awarie = historia[start:start + porcja]
    wektory = get_embeddings([ticket_to_text(a) for a in awarie])
    qdrant_client.upsert(
        collection_name=QDRANT_COLLECTION_NAME,
        points=[
            PointStruct(
                id=HISTORY_ID_OFFSET + start + i,
                vector=wektor,
                payload=awaria,
            )
            for i, (awaria, wektor) in enumerate(zip(awarie, wektory))
        ],
    )
    print(f"Załadowane: {start + len(awarie)}/{len(historia)}")

Załadowane: 50/150


Załadowane: 100/150


Załadowane: 150/150


In [19]:
# W kolekcji są też zgłoszenia z aplikacji, więc liczę tylko rekordy historii
TYLKO_HISTORIA = Filter(must=[FieldCondition(key="zrodlo", match=MatchValue(value="historia"))])

qdrant_client.count(collection_name=QDRANT_COLLECTION_NAME, count_filter=TYLKO_HISTORIA, exact=True).count

150

Szybka próba, czy wyszukiwanie ma sens - pytam o awarię opisaną własnymi słowami i patrzę, co wraca z historii.

In [20]:
results = qdrant_client.query_points(
    collection_name=QDRANT_COLLECTION_NAME,
    query=get_embeddings(["Wentylator nawiewny stanął i spada nadciśnienie w śluzie"])[0],
    # tak samo jak w aplikacji: szukam tylko w historii
    query_filter=TYLKO_HISTORIA,
    limit=5,
).points

pd.DataFrame([
    {
        "wynik": round(r.score, 3),
        "urzadzenie": r.payload["urzadzenie"],
        "objaw": r.payload["objaw"],
        "przyczyna": r.payload.get("przyczyna", ""),
    }
    for r in results
])

,wynik,urzadzenie,objaw,przyczyna
0,0.657,wentylator nawiewny V-15,"Wentylator nawiewny stanął, w pomieszczeniu cz...",Zwarcie w uzwojeniu silnika wentylatora.
1,0.614,wentylator nawiewny V-10,"Wentylator buczy i ma wyczuwalne drgania, wyda...",Zużyte łożysko wentylatora od strony napędu.
2,0.600,wentylator V-06,"Wentylator wyje i traci wydatek, ciśnienie w p...",Zatarte łożysko z powodu braku smarowania.
3,0.586,wentylator wyciągowy V-09,"Wentylator wyciągu procesowego stanął, w pomie...",Zablokowany wirnik i przeciążenie silnika.
4,0.568,kaskada ciśnień K-01,Różnica ciśnień między śluzą a pomieszczeniem ...,"Uszkodzony przetwornik różnicy ciśnień, błędne..."
